# Problem 4 (100 points)

Activation functions are the nonlinearities that give neural networks their approximation power. The choice of activation directly affects whether gradients vanish, explode, or flow cleanly during training. In this problem, you will implement activations from scratch, derive their gradients, diagnose the **dying ReLU** and **vanishing gradient** failure modes, and experimentally compare gradient flow across deep networks.

We use the following notation in this problem.
- $\sigma(x) = \frac{1}{1+e^{-x}}$ — sigmoid function.
- $\text{ReLU}(x) = \max(0, x)$.
- $\text{LReLU}_{\alpha}(x) = \max(\alpha x, x)$ — leaky ReLU with negative slope $\alpha$.
- $\text{GELU}(x) = x \cdot \Phi(x)$ where $\Phi$ is the standard normal CDF.
- $\text{Swish}(x) = x \cdot \sigma(x)$.
- $\sigma'(x)$ denotes $\frac{d\sigma}{dx}$.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (10 points, coding task)

**Do the following tasks.**

Implement each activation function **from scratch** using only basic torch arithmetic (`torch.exp`, `torch.clamp`, `torch.erf`, etc.).

1. `my_sigmoid(x)` — $\sigma(x) = \frac{1}{1+e^{-x}}$.
2. `my_relu(x)` — $\text{ReLU}(x) = \max(0, x)$.
3. `my_tanh(x)` — $\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$.
4. `my_leaky_relu(x, alpha=0.01)` — $\max(\alpha x, x)$.
5. `my_gelu(x)` — $x \cdot \Phi(x)$ using $\Phi(x) = \frac{1}{2}[1 + \text{erf}(x / \sqrt{2})]$.

All functions must preserve the input tensor shape.

- **Constraint**: Do not use `torch.sigmoid`, `torch.relu`, `torch.tanh`, `F.leaky_relu`, `F.gelu`, or any `nn` activation module.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def my_sigmoid(x):
    ...

def my_relu(x):
    ...

def my_tanh(x):
    ...

def my_leaky_relu(x, alpha=0.01):
    ...

def my_gelu(x):
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
x_test = torch.linspace(-3, 3, 100)
assert torch.allclose(my_sigmoid(x_test), torch.sigmoid(x_test), atol=1e-6), "Sigmoid mismatch"
assert torch.allclose(my_relu(x_test), torch.relu(x_test), atol=1e-6), "ReLU mismatch"
assert torch.allclose(my_tanh(x_test), torch.tanh(x_test), atol=1e-6), "Tanh mismatch"
assert torch.allclose(my_leaky_relu(x_test), F.leaky_relu(x_test, 0.01), atol=1e-6), "LeakyReLU mismatch"
assert torch.allclose(my_gelu(x_test), F.gelu(x_test), atol=1e-4), "GELU mismatch"
# Shape preservation
x_batch = torch.randn(8, 16)
assert my_sigmoid(x_batch).shape == (8, 16)
print("Part 1 passed!")

Now that we can compute each activation, let us derive their derivatives analytically.

## Part 2 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

Derive the derivative of each activation function.

1. Show that $\sigma'(x) = \sigma(x)(1 - \sigma(x))$. What is $\max_x \sigma'(x)$ and at which $x$ does it occur?
2. Show that $\tanh'(x) = 1 - \tanh^2(x)$. Evaluate $\tanh'(0)$.
3. What is $\text{ReLU}'(x)$ for $x \neq 0$? Why is the derivative at $x = 0$ technically undefined, and what convention do deep learning frameworks adopt?
4. Compute $\text{Swish}'(x) = \frac{d}{dx}[x \cdot \sigma(x)]$. Express the result in terms of $\sigma(x)$ and $x$.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Let us now implement these derivatives as code and verify them against PyTorch autograd.

## Part 3 (10 points, coding task)

**Do the following tasks.**

Implement the **analytical** derivative of each activation function.

1. `sigmoid_derivative(x)` — $\sigma(x)(1 - \sigma(x))$
2. `relu_derivative(x)` — $\mathbb{1}[x > 0]$ (returns float tensor)
3. `tanh_derivative(x)` — $1 - \tanh^2(x)$
4. `swish_derivative(x)` — $\sigma(x) + x \cdot \sigma(x)(1 - \sigma(x))$

Each function takes a tensor `x` and returns a tensor of the same shape.

- **Constraint**: Do not call `torch.autograd` inside these functions.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def sigmoid_derivative(x):
    ...

def relu_derivative(x):
    ...

def tanh_derivative(x):
    ...

def swish_derivative(x):
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
x = torch.linspace(-3, 3, 50, requires_grad=True)

# Sigmoid
torch.sigmoid(x).sum().backward()
assert torch.allclose(sigmoid_derivative(x.detach()), x.grad, atol=1e-5), "Sigmoid derivative mismatch"
x.grad.zero_()

# Tanh
torch.tanh(x).sum().backward()
assert torch.allclose(tanh_derivative(x.detach()), x.grad, atol=1e-5), "Tanh derivative mismatch"
x.grad.zero_()

# Swish
(x * torch.sigmoid(x)).sum().backward()
assert torch.allclose(swish_derivative(x.detach()), x.grad, atol=1e-5), "Swish derivative mismatch"

print("Part 3 passed!")

Visualizing activations and their derivatives together reveals where gradients vanish.

## Part 4 (10 points, coding task)

**Do the following tasks.**

Create a figure with **2 rows and 4 columns** (8 subplots).

- **Row 1**: Plot sigmoid, ReLU, tanh, and Swish over $[-5, 5]$.
- **Row 2**: Plot the derivative of each function over the same range.

For each subplot:
- Title with the function name.
- Dashed horizontal line at $y = 0$.
- In derivative plots, shade the region where $|f'(x)| < 0.1$ using `plt.fill_between` with `alpha=0.2, color='red'`. This highlights the **vanishing gradient zone**.

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

The plots show that sigmoid and tanh derivatives are small for large $|x|$. Let us quantify how this compounds through many layers.

## Part 5 (15 points, non-coding task)

**Do the following tasks (Reasoning is required).**

**Vanishing gradient analysis.**

In a network with $L$ layers, each using activation $\sigma$, the gradient at layer $l$ includes the product:

$$\prod_{k=l+1}^{L} \sigma'(z^{[k]})$$

1. The maximum value of $\sigma'(z)$ for sigmoid is $\frac{1}{4}$. Derive an upper bound for the gradient product across $L - l$ layers.
2. For a 10-layer sigmoid network, what is the upper bound on the gradient magnitude reaching layer 1 from layer 10?
3. Repeat the analysis for tanh. Is the vanishing gradient problem better or worse compared to sigmoid? Justify.
4. Why does ReLU largely solve the vanishing gradient problem? What is the gradient product through $L$ layers when all pre-activations $z^{[k]}$ are positive?
5. Name one scenario where ReLU is **worse** than sigmoid for gradient flow, and explain.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

The **dying ReLU** problem occurs when a neuron's pre-activation is negative for all inputs, making its gradient permanently zero. Let us demonstrate this experimentally.

## Part 6 (10 points, coding task)

**Do the following tasks.**

1. Build a 3-layer MLP: `Linear(10, 128) -> ReLU -> Linear(128, 128) -> ReLU -> Linear(128, 1)`.
2. Initialize **all biases** to $-2.0$ (this will force many neurons into the dead zone).
3. Pass `x = torch.randn(100, 10)` through the first two ReLU layers.
4. For each hidden layer, compute the **fraction of dead neurons**: neurons whose output is identically 0 for all 100 inputs.
5. Store as `dead_frac_layer1` and `dead_frac_layer2` (floats between 0 and 1).

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert 0 <= dead_frac_layer1 <= 1, "Dead fraction must be between 0 and 1"
assert 0 <= dead_frac_layer2 <= 1, "Dead fraction must be between 0 and 1"
assert dead_frac_layer1 > 0.3, f"Expected significant dying with bias=-2, got {dead_frac_layer1:.2%}"
print(f"Part 6 passed! Dead neurons: L1={dead_frac_layer1:.1%}, L2={dead_frac_layer2:.1%}")

Leaky ReLU and ELU were designed to fix the dying ReLU problem by allowing a small gradient when $x < 0$.

## Part 7 (5 points, non-coding task)

**Do the following tasks (Reasoning is not required).**

Fill in every cell of this comparison table:

| Activation | Range | $f'(0)$ | Vanishing gradient? | Dying neuron? | Zero-centered? |
|---|---|---|---|---|---|
| Sigmoid | ? | ? | ? | ? | ? |
| Tanh | ? | ? | ? | ? | ? |
| ReLU | ? | ? | ? | ? | ? |
| Leaky ReLU | ? | ? | ? | ? | ? |
| GELU | ? | ? | ? | ? | ? |
| Swish | ? | ? | ? | ? | ? |

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Let us now experimentally confirm the vanishing gradient phenomenon by measuring gradient magnitudes layer by layer.

## Part 8 (15 points, coding task)

**Do the following tasks.**

**Gradient flow experiment.**

1. Build three 20-layer MLPs: `Linear(1, 64)` followed by 19 layers of `Linear(64, 64)`, then `Linear(64, 1)`. Use Xavier initialization (`nn.init.xavier_uniform_`) for all weights.
   - `model_sig`: sigmoid activation at every hidden layer.
   - `model_tanh`: tanh activation at every hidden layer.
   - `model_relu`: ReLU activation at every hidden layer.
2. Pass `x = torch.randn(32, 1)` through each model, compute MSE loss against random targets, and call `.backward()`.
3. For each model, record the gradient norm $\|\nabla_{W^{[l]}} L\|_2$ at every layer $l$. Store as lists `grad_norms_sig`, `grad_norms_tanh`, `grad_norms_relu`, each of length 21.
4. Plot all three gradient norm curves on one plot with a **log scale** y-axis. Label each curve.

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert len(grad_norms_sig) == 21, f"Expected 21 gradient norms, got {len(grad_norms_sig)}"
assert len(grad_norms_relu) == 21
# Sigmoid gradients at layer 0 should be much smaller than at layer 20
assert grad_norms_sig[0] < grad_norms_sig[-1], "Sigmoid should show vanishing gradients at early layers"
print("Part 8 passed!")
print(f"Sigmoid  - first layer grad norm: {grad_norms_sig[0]:.2e}, last: {grad_norms_sig[-1]:.2e}")
print(f"ReLU     - first layer grad norm: {grad_norms_relu[0]:.2e}, last: {grad_norms_relu[-1]:.2e}")

Finally, let us study the GELU activation used in modern Transformers.

## Part 9 (15 points, coding task)

**Do the following tasks.**

The GELU activation has an exact form and a widely-used tanh approximation:

$$\text{GELU}_{\text{exact}}(x) = x \cdot \Phi(x) = \frac{x}{2}\left[1 + \text{erf}\!\left(\frac{x}{\sqrt{2}}\right)\right]$$

$$\text{GELU}_{\text{approx}}(x) \approx \frac{x}{2}\left[1 + \tanh\!\left(\sqrt{\frac{2}{\pi}}\left(x + 0.044715\, x^3\right)\right)\right]$$

1. Implement `gelu_exact(x)` using `torch.erf`.
2. Implement `gelu_approx(x)` using `torch.tanh`.
3. Compute and print the maximum absolute error between the two over $[-5, 5]$.
4. Use `torch.autograd.grad` to compute the derivative of `gelu_exact` and plot it over $[-5, 5]$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def gelu_exact(x):
    ...

def gelu_approx(x):
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
x = torch.linspace(-5, 5, 1000)
max_err = (gelu_exact(x) - gelu_approx(x)).abs().max().item()
assert max_err < 0.02, f"GELU approximation error too large: {max_err}"
gelu_builtin = F.gelu(x)
assert torch.allclose(gelu_exact(x), gelu_builtin, atol=1e-5), "GELU exact doesn't match PyTorch"
print(f"Part 9 passed! Max approximation error: {max_err:.6f}")